# 🏠 House Prices — 01. EDA & Feature Engineering
**Responsável:** Vitor  
**Dataset:** [Kaggle - House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)  
**Objetivo:** Este notebook centraliza a preparação dos dados (análise exploratória básica, imputação de valores faltantes, codificação de variáveis categóricas e normalização) para que os demais modelos (Regressão, Classificação, Clusterização) possam consumir um conjunto de dados padronizado e livre de inconsistências.

---

## 📦 0. Importações

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Configurações de visualização no Pandas
pd.set_option('display.max_columns', None)

print('✅ Bibliotecas importadas com sucesso!')

✅ Bibliotecas importadas com sucesso!


---
## 📂 1. Carregamento dos Dados

Carregando os dados brutos de treino localizados na pasta `data`.

In [2]:
# Caminho para os dados
data_path = '../data/train.csv'

# Verifica existência
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Arquivo não encontrado em {data_path}. Baixe do Kaggle e coloque na pasta.")

df = pd.read_csv(data_path)
print(f"Shape do dataset original: {df.shape}")
df.head(3)

Shape do dataset original: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500


---
## 🔍 2. Análise Exploratória Rápida

Identificamos as variáveis numéricas e a variável alvo (`SalePrice`).  
O `Id` é removido por ser apenas um identificador, sem valor preditivo.

As variáveis **categóricas** são mantidas no dataset exportado em sua forma original (cada notebook downstream realiza o encoding específico para seu modelo).

In [3]:
TARGET = 'SalePrice'
ID_COL = 'Id'

# Separa target e Id
y = df[TARGET]
X = df.drop(columns=[TARGET, ID_COL])

# Identificar colunas numéricas e categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numéricas : {len(numeric_features)} colunas')
print(f'Categóricas: {len(categorical_features)} colunas')
print(f'Valores nulos nas numéricas: {X[numeric_features].isnull().sum().sum()}')
print(f'Valores nulos nas categóricas: {X[categorical_features].isnull().sum().sum()}')

Numéricas : 36 colunas
Categóricas: 43 colunas
Valores nulos nas numéricas: 348
Valores nulos nas categóricas: 7481


C:\Users\vdmas\AppData\Local\Temp\ipykernel_41356\3768304249.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


---
## 🔧 3. Feature Engineering

Aplicamos as seguintes transformações:

**Variáveis numéricas:**
1. **Imputação** (`SimpleImputer`) — nulos substituídos pela **mediana** da coluna.
2. **Normalização** (`StandardScaler`) — features em escala padronizada (média 0, desvio 1).

**Variáveis categóricas:**
1. **Imputação** — nulos substituídos pela **moda** (valor mais frequente) da coluna.
2. Mantidas no formato original (sem encoding) — cada notebook aplica o encoding adequado ao seu modelo.

In [4]:
# Pipeline para variáveis numéricas: imputação + normalização
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Aplica nas numéricas
X_num_processed = numeric_transformer.fit_transform(X[numeric_features])
X_num_df = pd.DataFrame(X_num_processed, columns=numeric_features, index=X.index)

# Categóricas: imputa nulos com a moda (sem encoding)
X_cat_df = X[categorical_features].copy()
for col in categorical_features:
    moda = X_cat_df[col].mode()[0]
    X_cat_df[col] = X_cat_df[col].fillna(moda)

print(f'Numéricas  — shape: {X_num_df.shape} | nulos: {X_num_df.isnull().sum().sum()}')
print(f'Categóricas — shape: {X_cat_df.shape} | nulos: {X_cat_df.isnull().sum().sum()}')
X_num_df.describe().round(3)

Numéricas  — shape: (1460, 36) | nulos: 0
Categóricas — shape: (1460, 43) | nulos: 0


,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,TotRmsAbvGrd,Fireplaces,GarageYrBlt,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
count,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000,1460.000
mean,-0.000,0.000,-0.000,0.000,0.000,0.000,0.000,-0.000,-0.000,-0.000,-0.000,0.000,0.000,-0.000,0.000,-0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.000,-0.000,-0.000,0.000,-0.000,0.000,0.000,-0.000,0.000,0.000,0.000,-0.000,0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
min,-0.873,-2.219,-0.924,-3.688,-4.113,-3.288,-1.689,-0.571,-0.973,-0.289,-1.284,-2.411,-2.144,-0.795,-0.120,-2.249,-0.820,-0.241,-2.842,-0.762,-3.515,-4.751,-2.780,-0.951,-3.276,-2.365,-2.213,-0.752,-0.704,-0.359,-0.116,-0.270,-0.069,-0.088,-1.969,-1.368
25%,-0.873,-0.448,-0.297,-0.795,-0.517,-0.572,-0.866,-0.571,-0.973,-0.289,-0.779,-0.597,-0.726,-0.795,-0.120,-0.735,-0.820,-0.241,-1.026,-0.762,-1.062,-0.211,-0.934,-0.951,-0.692,-1.027,-0.648,-0.752,-0.704,-0.359,-0.116,-0.270,-0.069,-0.088,-0.489,-0.614
50%,-0.163,-0.039,-0.104,-0.072,-0.517,0.057,0.443,-0.571,-0.132,-0.289,-0.203,-0.150,-0.196,-0.795,-0.120,-0.098,-0.820,-0.241,0.790,-0.762,0.164,-0.211,-0.319,0.600,0.059,0.312,0.033,-0.752,-0.327,-0.359,-0.116,-0.270,-0.069,-0.088,-0.119,0.139
75%,0.310,0.415,0.109,0.651,0.382,0.952,0.927,0.338,0.589,-0.289,0.545,0.549,0.592,0.873,-0.120,0.497,1.108,-0.241,0.790,1.228,0.164,-0.211,0.297,0.600,0.934,0.312,0.482,0.589,0.322,-0.359,-0.116,-0.270,-0.069,-0.088,0.621,0.892
max,3.148,11.042,20.518,2.821,3.079,1.283,1.218,8.285,11.406,8.852,4.004,11.521,9.133,3.937,11.648,7.856,4.963,8.139,2.606,3.217,6.295,8.869,4.605,3.704,1.309,2.989,4.422,6.088,7.554,8.675,17.217,8.341,18.306,31.165,2.101,1.645


---
## 💾 4. Exportação do Dataset Processado

Exportamos um dataset unificado contendo:
- `Id` — identificador original
- **Variáveis numéricas** — imputadas e normalizadas
- **Variáveis categóricas** — no formato original (sem encoding)
- `SalePrice` — variável alvo em escala original

Este arquivo é utilizado pelos notebooks subsequentes como ponto de partida padronizado.

In [5]:
# Monta o dataset final
df_out = X_num_df.copy()

# Adiciona as categóricas originais
for col in categorical_features:
    df_out[col] = X_cat_df[col].values

# Reinsere Id e SalePrice
df_out.insert(0, ID_COL, df[ID_COL].values)
df_out[TARGET] = y.values

out_path = '../data/train_processed.csv'
df_out.to_csv(out_path, index=False)

print(f'✅ Dataset processado salvo em: {out_path}')
print(f'   Shape final: {df_out.shape}')
print(f'   Colunas numéricas (normalizadas): {len(numeric_features)}')
print(f'   Colunas categóricas (originais) : {len(categorical_features)}')
print(f'   Valores nulos: {df_out.isnull().sum().sum()}')

✅ Dataset processado salvo em: ../data/train_processed.csv
   Shape final: (1460, 81)
   Colunas numéricas (normalizadas): 36
   Colunas categóricas (originais) : 43
   Valores nulos: 0
